# 02 - does the 19 nt downstream window help?

`probe_rnafm.ipynb` (notebook 01) ran the 90 nt window. This one runs the **90 + 19 nt**
window and compares them **paired on identical folds and rows**, so the difference is not
confounded by the split.

Embeddings for the 109 nt window were extracted by `_run_window109.py`; this notebook loads
the cached arrays.

In [2]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, wilcoxon
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.simplefilter("ignore")
REPO = Path.cwd().parent if Path.cwd().name == "probing" else Path.cwd()
EMB, RES = REPO / "probing/embeddings", REPO / "probing/results"
ALPHAS = np.logspace(-3, 6, 40)

FOLDS = json.loads((RES / "folds.json").read_text())
print("folds:", FOLDS["n_folds"], "| seed:", FOLDS["seed"], "| schemes:", list(FOLDS["schemes"]))

folds: 10 | seed: 42 | schemes: ['KFold', 'GroupKFold']


In [3]:
def per_fold(make_model, X, y, folds):
    """Fit on each training fold, score the held-out fold. Returns (r2 array, rho array)."""
    r2, rho = [], []
    for tr, te in folds:
        tr, te = np.array(tr), np.array(te)
        m = make_model()
        m.fit(X[tr], y[tr])
        p = m.predict(X[te])
        r2.append(r2_score(y[te], p))
        rho.append(spearmanr(y[te], p).statistic)
    return np.array(r2), np.array(rho)

RIDGE = lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS))

def one_hot(seqs, L):
    idx = {c: i for i, c in enumerate("ACGU")}
    M = np.zeros((len(seqs), L, 4), dtype=np.float32)
    for i, s in enumerate(seqs):
        for j, c in enumerate(s):
            M[i, j, idx[c]] = 1.0
    return M.reshape(len(seqs), -1)

In [4]:
d90  = pd.read_csv(REPO/"probing/data/chen2013_window_up90.csv").rename(columns={"window":"sequence","strength":"Ts"})
d109 = pd.read_csv(REPO/"probing/data/chen2013_window_up90_dn19.csv").rename(columns={"window":"sequence","strength":"Ts"})
assert d90.name.equals(d109.name), "row order differs -> folds would not be comparable"
y = np.log(d90.Ts.to_numpy())                       # target: ln(Ts)
print("n =", len(y), "| 90 nt and 109 nt CSVs are row-aligned")

n = 582 | 90 nt and 109 nt CSVs are row-aligned


In [5]:
def reps(L, tag):
    st = {l: np.load(EMB / (f"rnafm_layer{l}_tokens.npy" if tag == "" else f"rnafm_L{L}_layer{l}_tokens.npy"))
          for l in [6, 9, 12]}
    R = {"A: L12 mean":  st[12][:, 1:L+1, :].mean(1),
         "B: L12 <cls>": st[12][:, 0, :],
         "C: L6 mean":   st[6][:, 1:L+1, :].mean(1),
         "D: L9 mean":   st[9][:, 1:L+1, :].mean(1)}
    return R

R90, R109 = reps(90, ""), reps(109, "L")
R90["E: one-hot"]  = one_hot(d90.sequence.tolist(), 90)
R109["E: one-hot"] = one_hot(d109.sequence.tolist(), 109)
for k in R90:
    print(f"{k:<14} 90nt {R90[k].shape}   109nt {R109[k].shape}")

A: L12 mean    90nt (582, 640)   109nt (582, 640)
B: L12 <cls>   90nt (582, 640)   109nt (582, 640)
C: L6 mean     90nt (582, 640)   109nt (582, 640)
D: L9 mean     90nt (582, 640)   109nt (582, 640)
E: one-hot     90nt (582, 360)   109nt (582, 436)


## Paired comparison, per arm with wilcoxon-rank test performed

In [10]:
print("R^2 comparing windows")
for scheme, folds in FOLDS["schemes"].items():
    print(f"=== target ln(Ts) | {scheme} ===")
    print(f"{'arm':<14}{'90 nt':>9}{'109 nt':>9}{'diff':>9}{'wins':>7}{'p':>8}{"sd-90nt":>9}{"sd-109nt":>9}")
    for k in R90:
        a, _ = per_fold(RIDGE, R90[k], y, folds)
        b, _ = per_fold(RIDGE, R109[k], y, folds)
        d = b - a
        print(f"{k:<14}{a.mean():>+9.3f}{b.mean():>+9.3f}{d.mean():>+9.3f}"
              f"{int((d>0).sum()):>5}/10{wilcoxon(a, b).pvalue:>8.3f}"
              f"{a.std():>9.3f}{b.std():>9.3f}")
    print()

R^2 comparing windows
=== target ln(Ts) | KFold ===
arm               90 nt   109 nt     diff   wins       p  sd-90nt sd-109nt
A: L12 mean      +0.691   +0.683   -0.008    4/10   0.492    0.042    0.041
B: L12 <cls>     +0.674   +0.693   +0.020    7/10   0.160    0.039    0.051
C: L6 mean       +0.668   +0.713   +0.045    7/10   0.037    0.047    0.048
D: L9 mean       +0.678   +0.699   +0.021    7/10   0.193    0.039    0.047
E: one-hot       +0.586   +0.652   +0.066   10/10   0.002    0.056    0.054

=== target ln(Ts) | GroupKFold ===
arm               90 nt   109 nt     diff   wins       p  sd-90nt sd-109nt
A: L12 mean      +0.595   +0.589   -0.006    5/10   1.000    0.147    0.148
B: L12 <cls>     +0.560   +0.600   +0.041    7/10   0.160    0.165    0.131
C: L6 mean       +0.573   +0.634   +0.061    9/10   0.014    0.132    0.131
D: L9 mean       +0.578   +0.610   +0.033    6/10   0.375    0.136    0.145
E: one-hot       +0.467   +0.552   +0.085    9/10   0.014    0.108    0.117



**Reading:** the 19 nt downstream carries real signal - one-hot gains most, and it is a purely
linear model on raw nucleotides, so the information is unambiguously present. But *mean-pooled
layer 12* is the one arm that cannot use it. That motivated the pooling ablation in notebook 03.